# 04 — Warehouse ELT, Dimensional Modeling, and Cloud Translation

## Business scenario

Analysts need stable SQL tables for daily revenue, customer behavior, and service
reliability. Silver Parquet is clean enough to reuse, but it is not yet a governed
business model.

This notebook uses DuckDB as a free local columnar analytical engine. DuckDB is not
Redshift/Snowflake or BigQuery; it lets us practice ELT SQL and inspect plans before translating
the design to a managed warehouse.

### Learning objectives

- Query partitioned Parquet without loading it into Pandas.
- Separate staging, dimensions, facts, and marts.
- Enforce grain and reconciliation checks.
- Build an SCD Type 2 customer history.
- inspect `EXPLAIN` output.
- Translate storage choices to BigQuery partitions/clusters, AWS sort/dist keys,
  and Snowflake clustering and micro-partition pruning.
- Understand how Airflow should orchestrate these jobs.


In [1]:
from pathlib import Path
import sys

import pandas as pd

try:
    import duckdb
except ImportError as exc:
    raise RuntimeError("Install requirements-core.txt before this lab") from exc

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
LAB_ROOT = PROJECT_ROOT / "lab_data"
SILVER_ORDERS = LAB_ROOT / "silver" / "orders"
SILVER_ITEMS = LAB_ROOT / "silver" / "order_items"
WAREHOUSE_PATH = LAB_ROOT / "warehouse" / "commerce.duckdb"
GOLD_ROOT = LAB_ROOT / "gold"
WAREHOUSE_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_ROOT.mkdir(parents=True, exist_ok=True)

if not list(SILVER_ORDERS.rglob("*.parquet")):
    raise FileNotFoundError("Silver Parquet not found. Run notebooks 01 and 02 first.")

connection = duckdb.connect(str(WAREHOUSE_PATH))
orders_glob = (SILVER_ORDERS / "**" / "*.parquet").as_posix()
items_glob = (SILVER_ITEMS / "**" / "*.parquet").as_posix()


## Define table grain before writing SQL

- `stg_orders`: one row per `order_id`.
- `stg_order_items`: one row per (`order_id`, `line_number`).
- `dim_customer`: one row per current customer.
- `dim_date`: one row per calendar date.
- `fact_order`: one row per order.
- `fact_order_item`: one row per order line.

Grain is a contract. If two rows unexpectedly represent the same order, sums can
double even though the SQL query succeeds.


In [2]:
connection.execute(f"""
    CREATE OR REPLACE TABLE stg_orders AS
    SELECT *
    FROM read_parquet('{orders_glob}', hive_partitioning = true)
""")
connection.execute(f"""
    CREATE OR REPLACE TABLE stg_order_items AS
    SELECT *
    FROM read_parquet('{items_glob}', hive_partitioning = true)
""")

staging_counts = connection.execute("""
    SELECT
        (SELECT COUNT(*) FROM stg_orders) AS order_rows,
        (SELECT COUNT(DISTINCT order_id) FROM stg_orders) AS distinct_orders,
        (SELECT COUNT(*) FROM stg_order_items) AS item_rows
""").df()
staging_counts


,order_rows,distinct_orders,item_rows
0,238,238,608


In [3]:
connection.execute("""
    CREATE OR REPLACE TABLE dim_customer AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY customer_id) AS customer_key,
        customer_id,
        MIN(event_timestamp) AS first_seen_at,
        MAX(updated_at) AS last_seen_at
    FROM stg_orders
    GROUP BY customer_id
""")

connection.execute("""
    CREATE OR REPLACE TABLE dim_date AS
    SELECT
        CAST(d AS DATE) AS date_key,
        EXTRACT(year FROM d)::INTEGER AS year,
        EXTRACT(month FROM d)::INTEGER AS month,
        EXTRACT(day FROM d)::INTEGER AS day,
        STRFTIME(d, '%A') AS day_name
    FROM GENERATE_SERIES(DATE '2026-01-01', DATE '2026-12-31', INTERVAL 1 DAY) t(d)
""")

connection.execute("""
    CREATE OR REPLACE TABLE fact_order AS
    SELECT
        o.order_id,
        c.customer_key,
        o.customer_id,
        CAST(o.event_date AS DATE) AS order_date,
        o.status,
        o.currency,
        o.total_amount,
        o.updated_at
    FROM stg_orders o
    JOIN dim_customer c USING (customer_id)
""")

connection.execute("""
    CREATE OR REPLACE TABLE fact_order_item AS
    SELECT
        i.order_id,
        i.line_number,
        i.sku,
        i.quantity,
        i.unit_price,
        i.quantity * i.unit_price AS line_amount,
        CAST(i.event_date AS DATE) AS order_date
    FROM stg_order_items i
""")


## Quality gates belong next to the model

Tests should describe business invariants, not merely confirm that a DataFrame
exists. These checks fail loudly when the declared grains or relationships break.


In [4]:
quality_checks = {
    "duplicate_order_ids": connection.execute(
        "SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM fact_order"
    ).fetchone()[0],
    "orphan_order_items": connection.execute("""
        SELECT COUNT(*)
        FROM fact_order_item i
        LEFT JOIN fact_order o USING (order_id)
        WHERE o.order_id IS NULL
    """).fetchone()[0],
    "missing_customer_keys": connection.execute(
        "SELECT COUNT(*) FROM fact_order WHERE customer_key IS NULL"
    ).fetchone()[0],
    "nonpositive_item_values": connection.execute(
        "SELECT COUNT(*) FROM fact_order_item WHERE quantity <= 0 OR unit_price < 0"
    ).fetchone()[0],
}
assert all(value == 0 for value in quality_checks.values()), quality_checks
quality_checks


{'duplicate_order_ids': 0,
 'orphan_order_items': 0,
 'missing_customer_keys': 0,
 'nonpositive_item_values': 0}

## Build a business mart and reconcile money

An order total and the sum of its lines can differ because of tax, shipping,
discounts, or bad source data. A data engineer should make the expected rule
explicit rather than assume equality silently. Here the source generator defines
total as the item sum, so we test a small decimal tolerance.


In [5]:
reconciliation = connection.execute("""
    WITH item_totals AS (
        SELECT order_id, SUM(line_amount) AS item_total
        FROM fact_order_item
        GROUP BY order_id
    )
    SELECT
        COUNT(*) FILTER (WHERE ABS(o.total_amount - i.item_total) > 0.01) AS mismatched_orders,
        MAX(ABS(o.total_amount - i.item_total)) AS maximum_difference
    FROM fact_order o
    JOIN item_totals i USING (order_id)
""").df()
assert reconciliation.loc[0, "mismatched_orders"] == 0
reconciliation


,mismatched_orders,maximum_difference
0,0,0.0


In [6]:
connection.execute("""
    CREATE OR REPLACE TABLE mart_daily_sales AS
    SELECT
        o.order_date,
        COUNT(*) AS order_count,
        COUNT(DISTINCT o.customer_id) AS purchasing_customers,
        SUM(o.total_amount) AS gross_revenue,
        AVG(o.total_amount) AS average_order_value,
        SUM(CASE WHEN o.status = 'shipped' THEN o.total_amount ELSE 0 END) AS shipped_revenue
    FROM fact_order o
    GROUP BY o.order_date
    ORDER BY o.order_date
""")
connection.execute("SELECT * FROM mart_daily_sales ORDER BY order_date LIMIT 10").df()


,order_date,order_count,purchasing_customers,gross_revenue,average_order_value,shipped_revenue
0,2026-01-01,83,49,26777.96,322.626024,9707.58
1,2026-01-02,85,50,25903.80,304.750588,8393.49
2,2026-01-03,70,41,21061.80,300.882857,6788.48


## Slowly changing dimension Type 2

SCD2 preserves attribute history rather than overwriting the old value. Each
customer version has an effective interval and one current row. The example uses
a customer tier change; in a real load, the transaction must make the expiration
and insertion atomic.


In [7]:
connection.execute("""
    CREATE OR REPLACE TABLE dim_customer_scd AS
    SELECT
        customer_key * 10 AS customer_version_key,
        customer_id,
        'standard'::VARCHAR AS customer_tier,
        TIMESTAMP '2026-01-01 00:00:00' AS effective_from,
        TIMESTAMP '9999-12-31 00:00:00' AS effective_to,
        TRUE AS is_current
    FROM dim_customer
""")

changed_customer = connection.execute(
    "SELECT customer_id FROM dim_customer ORDER BY customer_id LIMIT 1"
).fetchone()[0]
effective_time = "2026-02-01 09:00:00"

connection.execute("BEGIN TRANSACTION")
try:
    connection.execute(
        """UPDATE dim_customer_scd
           SET effective_to = ?::TIMESTAMP, is_current = FALSE
           WHERE customer_id = ? AND is_current = TRUE""",
        [effective_time, changed_customer],
    )
    next_key = connection.execute(
        "SELECT COALESCE(MAX(customer_version_key), 0) + 1 FROM dim_customer_scd"
    ).fetchone()[0]
    connection.execute(
        """INSERT INTO dim_customer_scd VALUES
           (?, ?, 'plus', ?::TIMESTAMP, TIMESTAMP '9999-12-31 00:00:00', TRUE)""",
        [next_key, changed_customer, effective_time],
    )
    connection.execute("COMMIT")
except Exception:
    connection.execute("ROLLBACK")
    raise

history = connection.execute(
    """SELECT customer_version_key, customer_id, customer_tier,
              CAST(effective_from AS VARCHAR) AS effective_from,
              CAST(effective_to AS VARCHAR) AS effective_to,
              is_current
       FROM dim_customer_scd
       WHERE customer_id = ?
       ORDER BY effective_from""",
    [changed_customer],
).df()
assert len(history) == 2 and history["is_current"].sum() == 1
history


,customer_version_key,customer_id,customer_tier,effective_from,effective_to,is_current
0,10,cust_0001,standard,2026-01-01 00:00:00,2026-02-01 09:00:00,False
1,601,cust_0001,plus,2026-02-01 09:00:00,9999-12-31 00:00:00,True


## Inspect the query plan

Managed MPP warehouses and DuckDB have different optimizers and execution
architectures. The transferable habit is to inspect the plan for scans, filters,
join strategy, and cardinality rather than tuning from folklore.


In [8]:
explain_rows = connection.execute("""
    EXPLAIN ANALYZE
    SELECT c.customer_id, SUM(o.total_amount) AS revenue
    FROM fact_order o
    JOIN dim_customer c USING (customer_key)
    WHERE o.order_date BETWEEN DATE '2026-01-01' AND DATE '2026-01-07'
    GROUP BY c.customer_id
    ORDER BY revenue DESC
    LIMIT 10
""").fetchall()
print("\n".join(str(row[1]) for row in explain_rows))


┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
     EXPLAIN ANALYZE     SELECT c.customer_id, SUM(o.total_amount) AS revenue     FROM fact_order o     JOIN dim_customer c USING (customer_key)     WHERE o.order_date BETWEEN DATE '2026-01-01' AND DATE '2026-01-07'     GROUP BY c.customer_id     ORDER BY revenue DESC     LIMIT 10 
┌────────────────────────────────────────────────┐
│┌──────────────────────────────────────────────┐│
││              Total Time: 0.0059s             ││
│└──────────────────────────────────────────────┘│
└────────────────────────────────────────────────┘
┌───────────────────────────┐
│           QUERY           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│      EXPLAIN_ANALYZE      │
│    ────────────────────   │
│                           │
│           0 rows          │
│           0.00s     

In [9]:
gold_path = (GOLD_ROOT / "mart_daily_sales.parquet").as_posix()
connection.execute(f"COPY mart_daily_sales TO '{gold_path}' (FORMAT PARQUET, COMPRESSION ZSTD)")
print(f"Published: {gold_path}")


Published: C:/Users/sungj/sam_data_engineering_tutoring_capstone/lab_data/gold/mart_daily_sales.parquet


In [10]:
import pandas as pd

gold_file = (
    PROJECT_ROOT
    / "lab_data"
    / "gold"
    / "mart_daily_sales.parquet"
)

daily_sales = pd.read_parquet(gold_file)
daily_sales

,order_date,order_count,purchasing_customers,gross_revenue,average_order_value,shipped_revenue
0,2026-01-01,83,49,26777.96,322.626024,9707.58
1,2026-01-02,85,50,25903.80,304.750588,8393.49
2,2026-01-03,70,41,21061.80,300.882857,6788.48


In [11]:
connection.execute("""
    SELECT *
    FROM read_parquet(?)
    ORDER BY order_date
""", [str(gold_file)]).df()

,order_date,order_count,purchasing_customers,gross_revenue,average_order_value,shipped_revenue
0,2026-01-01,83,49,26777.96,322.626024,9707.58
1,2026-01-02,85,50,25903.80,304.750588,8393.49
2,2026-01-03,70,41,21061.80,300.882857,6788.48


## Translate the design to BigQuery and Redshift/Snowflake

### BigQuery

- Partition `fact_order` by `order_date`.
- Cluster by frequently filtered/joined columns such as `customer_id` and `status`.
- Preview estimated bytes before running a query.
- Use the no-credit-card BigQuery sandbox for a small real-cloud exercise.

See `sql/bigquery_reference.sql`.

### Redshift/Snowflake

- On the AWS platform, choose `DISTKEY` or `SORTKEY` from workload evidence.
- On Snowflake, inspect automatic micro-partition pruning before adding clustering keys.
- Load columnar files through each platform's `COPY` or staged-file workflow.
- Analyze query plans and table statistics.
- A local DuckDB run does not prove Redshift/Snowflake administration experience.

See `sql/redshift_reference.sql` for AWS-specific DDL and adapt the concepts rather
than copying that syntax into Snowflake. Redshift/Snowflake cloud trials are optional,
time-limited exercises because eligibility and cost controls differ from the core free lab.


## Airflow orchestration

The included `dags/commerce_pipeline_dag.py` demonstrates:

- a daily logical data interval;
- catchup/backfill;
- bounded task retries;
- small metadata passed between tasks;
- a quality gate that fails visibly;
- persisted run metrics.

Start it with `docker compose -f docker-compose.airflow.yml up`. Airflow should
orchestrate versioned jobs; production transformations should not live only inside
notebook cells.

### Your turn

1. Add a uniqueness failure and confirm the quality gate blocks publication.
2. Make the SCD2 change idempotent for a repeated change event.
3. Add a `fact_application_error` table from the Silver logs.
4. Port `mart_daily_sales` to BigQuery sandbox and compare SQL differences.
5. Explain how you would backfill only `2026-01-07` in Airflow.
